# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Randaadad/FlyRank-AI/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Rule

I will rank pages for review using two observable signals: search visibility and content staleness.

Pages receive higher scores when they have meaningful search exposure and have not been updated recently. The rule is a directional review queue, not a claim that refreshing a page will cause recovery.

The score is:

- +2 points when impressions_90d >= 500
- +2 points when days_since_last_update >= 180
- +1 point when impressions_90d >= 250
- +1 point when days_since_last_update >= 90

The reason code is assigned from the strongest triggered signal:

- `stale_visible_page` — the page has at least 500 impressions and has not been updated for at least 180 days.
- `stale_page` — the page has not been updated for at least 180 days.
- `visible_page` — the page has at least 500 impressions.
- `low_priority_review` — the page does not meet the stronger thresholds.

The action label is `REVIEW_REFRESH` for pages with a score of at least 2 and `NO_ACTION` otherwise.

In [10]:
import os
import pandas as pd
import numpy as np

# Show where the notebook is currently running
print("Current directory:")
print(os.getcwd())

# Search for the starter CSV
matches = []

for root, dirs, files in os.walk("."):
    for file in files:
        if file == "content_refresh_anonymized.csv":
            matches.append(os.path.join(root, file))

print("\nFound files:")
for path in matches:
    print(path)

Current directory:
/content

Found files:


## 2. Build the ranked queue (writes the CSV)


*   List item
*   List item


The score prioritizes pages where there is both enough observed visibility to matter and evidence of staleness. The queue is intended for human review, not automatic editing.

In [11]:
import duckdb
from google.colab import userdata
import pandas as pd
import numpy as np
import os

# Connect to DuckDB
con = duckdb.connect()

# Get Hugging Face token
HF_TOKEN = userdata.get("HF_TOKEN")

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    )
    """
)

# Load March 2026 daily performance data
df = con.sql("""
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
""").df()

print("Rows loaded:", len(df))
print("Columns:", len(df.columns))

# --------------------------------------------------
# Build baseline score
# --------------------------------------------------

# Start score at 0
df["baseline_score"] = 0

# Visibility signal
df.loc[df["impressions"] >= 500, "baseline_score"] += 2

# Moderate visibility
df.loc[
    (df["impressions"] >= 250) &
    (df["impressions"] < 500),
    "baseline_score"
] += 1

# Engagement signal
df.loc[df["clicks"] >= 20, "baseline_score"] += 1

# Create reason code
def get_reason(row):

    if (
        row["impressions"] >= 500
        and row["clicks"] >= 20
    ):
        return "high_visibility_engagement"

    elif row["impressions"] >= 500:
        return "high_visibility"

    elif row["clicks"] >= 20:
        return "engagement_signal"

    else:
        return "low_priority_review"


df["reason_code"] = df.apply(get_reason, axis=1)

# Action label
df["action_label"] = np.where(
    df["baseline_score"] >= 2,
    "REVIEW",
    "NO_ACTION"
)

# Rank the queue
df = df.sort_values(
    ["baseline_score", "impressions"],
    ascending=[False, False]
).reset_index(drop=True)

df["rank"] = np.arange(1, len(df) + 1)

# Select output columns
queue = df[
    [
        "rank",
        "content_hash_id",
        "client_hash_id",
        "baseline_score",
        "reason_code",
        "action_label",
        "impressions",
        "clicks"
    ]
].copy()

# --------------------------------------------------
# Write CSV
# --------------------------------------------------

os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"

queue.to_csv(output_path, index=False)

print("CSV written successfully:")
print(output_path)

print("\nTop 20:")
display(queue.head(20))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows loaded: 9841378
Columns: 31


KeyError: 'impressions'

## 3. Top-20 review

The top 20 pages are reviewed as decision-support candidates. The confidence note describes why the available evidence supports the ranking. The "what would make it wrong" note identifies a plausible alternative explanation or data limitation.

In [ ]:
top20 = queue.head(20).copy()

def confidence_note(row):
    if (
        row["impressions_90d"] >= 500
        and row["days_since_last_update"] >= 180
    ):
        return "High evidence: meaningful visibility and strong staleness signal."

    if row["impressions_90d"] >= 500:
        return "Moderate evidence: strong visibility, but staleness is weaker."

    if row["days_since_last_update"] >= 180:
        return "Moderate evidence: strong staleness signal, but lower observed visibility."

    return "Lower evidence: the page does not meet both stronger thresholds."


def wrong_if(row):
    if (
        row["impressions_90d"] >= 500
        and row["days_since_last_update"] >= 180
    ):
        return "Wrong if the page is intentionally evergreen or its low freshness is not a meaningful improvement opportunity."

    if row["impressions_90d"] >= 500:
        return "Wrong if the page's current visibility does not represent a realistic refresh opportunity."

    if row["days_since_last_update"] >= 180:
        return "Wrong if the page has too little search demand for a refresh to matter."

    return "Wrong if the observed signals are too weak to justify review."


top20["confidence_note"] = top20.apply(confidence_note, axis=1)
top20["what_would_make_it_wrong"] = top20.apply(wrong_if, axis=1)

review = top20[
    [
        "rank",
        "content_id",
        "action_label",
        "reason_code",
        "baseline_score",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

display(review)

## 4. Weak picks + leakage check


I will inspect the lowest-scoring pages that still receive `REVIEW_REFRESH`. These are useful because they show where the rule may be too broad.


The baseline uses only decision-time observable fields: impressions, content age, and days since last update. It does not use the starter target `trend_direction`, future-window outcomes, product flags, model probabilities, or private client information.

The starter dataset's `trend_direction` is a proxy label and should not be used as an input to this baseline. :contentReference[oaicite:3]{index=3}

In [ ]:
# Weakest pages that are still recommended for review
weak_picks = (
    queue[queue["action_label"] == "REVIEW_REFRESH"]
    .sort_values(
        ["baseline_score", "impressions_90d"],
        ascending=[True, True]
    )
    .head(10)
)

print("Weakest review picks:")
display(weak_picks)


# -----------------------------
# Leakage check
# -----------------------------

forbidden_columns = [
    "trend_direction",
    "is_declining_label",
    "best_model_probability",
    "final_refresh_score"
]

present_forbidden = [
    col for col in forbidden_columns
    if col in queue.columns
]

print("\nLeakage check:")
print("Forbidden columns present in queue:", present_forbidden)

assert len(present_forbidden) == 0, (
    f"Potential leakage columns found: {present_forbidden}"
)

print("PASS: no target/model/future outcome columns are used in the queue.")

# Confirm required observable inputs exist
required_inputs = [
    "impressions_90d",
    "days_since_last_update",
    "content_age_days"
]

missing_inputs = [
    col for col in required_inputs
    if col not in df.columns
]

assert len(missing_inputs) == 0, (
    f"Missing required inputs: {missing_inputs}"
)

print("PASS: all baseline inputs are available.")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.